In [1]:
from diffusion_policy.model.denoiser import ConditionalUnet1D
from diffusion_policy.model.visual_encoder import get_visual_encoder
import torch

Diffusion Denoiser/Noise Predictor $\epsilon_{\theta}(A_t, k)$

In [4]:
batch_size, action_pred_horizon, action_dim = 2, 20, 4
x = torch.rand((batch_size, action_pred_horizon, action_dim))


obs_horizon, obs_dim = 4, 16
cond_dim = obs_horizon * obs_dim
cond = torch.rand((batch_size, cond_dim))

diff_step = torch.randint(0, 100, (batch_size,))

denoiser = ConditionalUnet1D(
    input_dim=action_dim,
    cond_dim=cond_dim,
    kernel_size=3,
    n_groups=2,
)
pred_noise = denoiser(x, cond)

print(pred_noise.shape)

Number of Parameters: 39,257,860
torch.Size([2, 256, 20])
torch.Size([2, 20, 4])


Visual Encoder

In [5]:
visual_encoder = get_visual_encoder()

images = torch.zeros((batch_size, obs_horizon, 3, 96, 96))
images = images.flatten(start_dim=0, end_dim=1)  # (batch_size * obs_horizon, 3, 96, 96)
print(images.shape)
image_embeds = visual_encoder(images)
print(image_embeds.shape)


torch.Size([8, 3, 96, 96])
torch.Size([8, 512])


## LIBERO 

In [1]:
from diffusion_policy.dataset.libero import (
    get_libero_dataset,
    get_hdf5_files_from_folders,
    preprocess_libero_batch,
)
from torch.utils.data import DataLoader
import torch

ROBOMIMIC WARNING(
    No private macro file found!
    It is recommended to use a private macro file
    To setup, run: python /home/richard/miniconda3/envs/diff_policy/lib/python3.13/site-packages/robomimic/scripts/setup_macros.py
)

============= Initialized Observation Utils with Obs Spec =============

using obs modality: low_dim with keys: ['joint_states', 'ee_pos', 'gripper_states', 'ee_ori']
using obs modality: rgb with keys: ['agentview_rgb', 'eye_in_hand_rgb']


Loading a single hdf5

In [5]:
hdf5_filename = "KITCHEN_SCENE4_put_the_black_bowl_in_the_bottom_drawer_of_the_cabinet_and_close_it_demo.hdf5"
example_hdf5 = f"../data/libero/libero_10/{hdf5_filename}"
hdf5_files = [example_hdf5]
action_pred_horizon = 16

ds = get_libero_dataset(
    hdf5_files=hdf5_files,
    split="train",
    seq_length=action_pred_horizon,
)

dl = DataLoader(
    ds,
    batch_size=8,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
)

print(f"number of training samples in dataset {len(ds)}")
batch = next(iter(dl))

SequenceDataset: loading dataset into memory...
100%|██████████| 45/45 [00:00<00:00, 1158.76it/s]
number of training samples in dataset 11174


In [ ]:
# Each batch is a dictionary
for k, v in batch.items():
    if k == "language":
        print(k, v)
        continue
    if k != "obs":
        print(k, v.shape)


print("== actions ==")
# they're already normalized
print(torch.min(batch["actions"]), torch.max(batch["actions"]))


print("== obs ==")
obs_dict = batch["obs"]
for obs_key, obs_tensor in obs_dict.items():
    print(
        obs_key,
        obs_tensor.dtype,
        obs_tensor.shape,
        torch.min(obs_tensor),
        torch.max(obs_tensor),
    )

print("== rewards (indicates success if 1) ==")
print(batch["rewards"])

actions torch.Size([8, 16, 7])
rewards torch.Size([8, 16])
language ['put the black bowl in the bottom drawer of the cabinet and close it', 'put the black bowl in the bottom drawer of the cabinet and close it', 'put the black bowl in the bottom drawer of the cabinet and close it', 'put the black bowl in the bottom drawer of the cabinet and close it', 'put the black bowl in the bottom drawer of the cabinet and close it', 'put the black bowl in the bottom drawer of the cabinet and close it', 'put the black bowl in the bottom drawer of the cabinet and close it', 'put the black bowl in the bottom drawer of the cabinet and close it']
== actions ==
tensor(-1.) tensor(0.8464)
== obs ==
agentview_rgb torch.uint8 torch.Size([8, 16, 128, 128, 3]) tensor(0, dtype=torch.uint8) tensor(255, dtype=torch.uint8)
ee_pos torch.float64 torch.Size([8, 16, 3]) tensor(-0.2180, dtype=torch.float64) tensor(1.1634, dtype=torch.float64)
ee_ori torch.float64 torch.Size([8, 16, 3]) tensor(-0.1613, dtype=torch.floa

In [9]:
# For training only
device = torch.device("cuda")
batch_processed = preprocess_libero_batch(
    batch, device, obs_horizon=2, image_keys=("agentview_rgb")
)

for k, v in batch_processed.items():
    if k == "language":
        print(k, v)
        continue
    if k != "obs":
        print(k, v.shape, v.device)

print("== obs ==")
obs_dict = batch_processed["obs"]
for obs_key, obs_tensor in obs_dict.items():
    print(
        obs_key,
        obs_tensor.dtype,
        obs_tensor.shape,
        torch.min(obs_tensor),
        torch.max(obs_tensor),
    )

actions torch.Size([8, 16, 7]) cuda:0
rewards torch.Size([8, 16]) cuda:0
language ['put the black bowl in the bottom drawer of the cabinet and close it', 'put the black bowl in the bottom drawer of the cabinet and close it', 'put the black bowl in the bottom drawer of the cabinet and close it', 'put the black bowl in the bottom drawer of the cabinet and close it', 'put the black bowl in the bottom drawer of the cabinet and close it', 'put the black bowl in the bottom drawer of the cabinet and close it', 'put the black bowl in the bottom drawer of the cabinet and close it', 'put the black bowl in the bottom drawer of the cabinet and close it']
== obs ==
agentview_rgb torch.float32 torch.Size([8, 2, 128, 3, 128]) tensor(0., device='cuda:0') tensor(0.0039, device='cuda:0')
joint_states torch.float64 torch.Size([8, 2, 7]) tensor(-2.4844, device='cuda:0', dtype=torch.float64) tensor(2.2561, device='cuda:0', dtype=torch.float64)
gripper_states torch.float64 torch.Size([8, 2, 2]) tensor(-0.03

Loading multiple hdf5 into a single dataset

In [11]:
# Get list of hdf5 file paths
hdf5_files = get_hdf5_files_from_folders(
    ["../data/libero/libero_10", "../data/libero/libero_goal"]
)
print(hdf5_files)

# Just pass this list to this
ds = get_libero_dataset(
    hdf5_files=hdf5_files,
    obs_keys=("agentview_rgb", "joint_states", "gripper_states"),
    split="train",
    seq_length=action_pred_horizon,
)

['../data/libero/libero_10/KITCHEN_SCENE3_turn_on_the_stove_and_put_the_moka_pot_on_it_demo.hdf5', '../data/libero/libero_10/KITCHEN_SCENE6_put_the_yellow_and_white_mug_in_the_microwave_and_close_it_demo.hdf5', '../data/libero/libero_10/LIVING_ROOM_SCENE2_put_both_the_alphabet_soup_and_the_tomato_sauce_in_the_basket_demo.hdf5', '../data/libero/libero_10/LIVING_ROOM_SCENE5_put_the_white_mug_on_the_left_plate_and_put_the_yellow_and_white_mug_on_the_right_plate_demo.hdf5', '../data/libero/libero_10/LIVING_ROOM_SCENE6_put_the_white_mug_on_the_plate_and_put_the_chocolate_pudding_to_the_right_of_the_plate_demo.hdf5', '../data/libero/libero_10/LIVING_ROOM_SCENE1_put_both_the_alphabet_soup_and_the_cream_cheese_box_in_the_basket_demo.hdf5', '../data/libero/libero_10/KITCHEN_SCENE8_put_both_moka_pots_on_the_stove_demo.hdf5', '../data/libero/libero_10/LIVING_ROOM_SCENE2_put_both_the_cream_cheese_box_and_the_butter_in_the_basket_demo.hdf5', '../data/libero/libero_10/KITCHEN_SCENE4_put_the_black_bo

Replaying a HDF5

In [10]:
import h5py
from IPython.display import HTML
import imageio
import numpy as np

demo_num = 1  # [0,50]

with h5py.File(example_hdf5, "r") as f:
    images = f[f"data/demo_{demo_num}/obs/agentview_rgb"][:]
    rewards = f[f"data/demo_{demo_num}/rewards"][:]
    pos = f[f"data/demo_{demo_num}/obs/ee_pos"][:]

    print(type(images), images.shape)
    print(type(rewards), rewards.shape)


np.set_printoptions(precision=2)
print(rewards)
print("ee_pos", pos)
video_writer = imageio.get_writer("output.mp4", fps=60)
for image in images:
    video_writer.append_data(image[::-1])
video_writer.close()

HTML("""
    <video width="640" height="480" controls>
        <source src="output.mp4" type="video/mp4">
    </video>
    <script>
        var video = document.getElementsByTagName('video')[0];
        video.playbackRate = 2.0; // Increase the playback speed to 2x
        </script>    
""")


<class 'numpy.ndarray'> (228, 128, 128, 3)
<class 'numpy.ndarray'> (228,)
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 1]
ee_pos [[-1.98e-01 -9.82e-03  1.19e+00]
 [-1.98e-01 -9.76e-03  1.19e+00]
 [-1.98e-01 -9.67e-03  1.19e+00]
 [-1.97e-01 -9.51e-03  1.19e+00]
 [-1.97e-01 -9.20e-03  1.19e+00]
 [-1.97e-01 -8.63e-03  1.19e+00]
 [-1.96e-01 -8.00e-03  1.19e+00]
 [-1.96e-01 -7.73e-03  1.19e+00]
 [-1.96e-01 -7.72e-03  1.19e+00]
 [-1.94e-01 -7.86e-03  1.19e+00]
 [-1.92e-01 -8.06e-03  1.19e+00]
 [-1.88e-01 -8.34e-03  1.19e+00]
 [-1.83e-01 -8.66e-03  1.19e+00]
 [-1.77e-01 -9.07e-03  1.1